# Price-return statistics

A closer statistical look at one ticker's daily returns, built on
[`src/tools/price_return/stats.py`](../src/tools/price_return/stats.py). Where
[`price_return_analysis_v0.5.ipynb`](price_return_analysis_v0.5.ipynb) counts streaks and rare
moves, this notebook asks what those counts rest on:

| Section | Question |
|---|---|
| 1 Distribution and tails | How far from normal are the returns, and how fat are the tails? What is the value at risk? |
| 2 Dependence | Is there memory in direction (trend or reversion), or in size (volatility clustering)? |
| 3 Drawdowns and risk | How deep and long were the drawdowns? How good was the return for the risk, and when? |
| 4 Rare-event uncertainty | How sure are the rare-event probabilities, and how far are they from what a normal or fat-tailed model predicts? |
| 5 Option sizing | Price bands from volatility, move probabilities, and the expected P&L of the old `test_es` grids |

Needs the `stats` extra (scipy): `pip install -e ".[stats]"`. Every return is a decimal
(0.01 = 1%) and every multi-day return is compounded.

In [ ]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with
# `pip install -e ".[stats]"`, so `from src.tools...` resolves from any working directory.
#
# On Colab the install still runs, for the dependencies - but an editable
# install registers itself through a .pth file that Python executes only at
# interpreter startup, so this already-running kernel would not see it and the
# import would fail with ModuleNotFoundError. Putting the repo root on sys.path
# makes it importable now, without restarting the runtime.
import os
import sys

REPO = '/content/drive/MyDrive/github/quant_dev'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        raise FileNotFoundError(
            f'No repo at {REPO}. Update REPO above to where quant_dev lives on '
            f'your Drive.')
    !pip install -q -e "{REPO}[stats]"
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

print(f'Running in Colab: {IN_COLAB}')

In [ ]:
import pandas as pd
from IPython.display import display

from src.tools.price_return import (
    Params, load_ticker_config, load_price_data, add_rolling_stats, daily_returns_series,
    latest_snapshot, consecutive_analysis,
    # 1. distribution and tails
    return_moments, jarque_bera, fit_student_t, tail_index, value_at_risk, plot_qq,
    # 2. dependence
    autocorrelation, ljung_box, variance_ratio, arch_lm, plot_autocorrelation,
    # 3. drawdowns and risk
    risk_ratios, drawdown_table, rolling_risk, plot_drawdown, plot_rolling_risk,
    # 4. rare-event uncertainty
    probability_intervals, model_probabilities, plot_event_probabilities,
    # 5. option sizing
    price_range, move_probabilities, expected_pnl,
)

pct = lambda values, digits=2: values.map(f'{{:.{digits}%}}'.format)

## Parameters

The ticker's thresholds come from `configs/tickers.yaml` when it is listed there, so the numbers
line up with `rare_case_run`. Any other ticker falls back to Yahoo data from 2016.

In [ ]:
TICKER = 'ES=F'
DRILL_N_DAYS = 3        # holding period for the rare-event uncertainty (section 4)
N_BOOT = 1000           # bootstrap resamples; fewer runs faster, with noisier interval ends
WINDOW = 250            # rolling window for section 3, in trading days

P = load_ticker_config(verbose=False).get(TICKER) or Params(
    ticker=TICKER, data_source='yahoo', start_date='2016-01-01')
# To try it offline: P = Params(data_source='simulated', start_date='2016-01-01')
P

In [ ]:
# Q: What series is this, and over what period?
df = add_rolling_stats(load_price_data(P), P)
r = daily_returns_series(df)
print(f'{P.ticker}: {len(r)} daily returns, {r.index[0]:%Y-%m-%d} to {r.index[-1]:%Y-%m-%d}')

## 1. Distribution and tails

In [ ]:
# Q: How far from normal are the daily returns?
# Excess kurtosis is 0 for a normal; daily market returns usually sit well above it
# (fat tails). A tiny Jarque-Bera p-value says the difference is not sampling noise.
moments, jb = return_moments(r), jarque_bera(r)
display(pd.concat([moments, jb[['statistic', 'p_value']].rename(
    {'statistic': 'Jarque-Bera statistic', 'p_value': 'Jarque-Bera p-value'})]).to_frame('value'))

In [ ]:
# Q: Does a fat-tailed Student-t describe the returns better than a normal?
# Points on the grey line are where a fit matches the data. Look at the ends: where the
# normal's points bend away and the t's stay closer, the t captures the tails.
# The fitted degrees of freedom say how fat: below about 5 is very fat-tailed.
display(fit_student_t(r).to_frame('Student-t fit'))
plot_qq(r).show()

In [ ]:
# Q: How fat is each tail?
# Hill tail index alpha from the largest 5% of moves on each side. Smaller is fatter:
# moments above alpha do not exist, so alpha below 4 means kurtosis is not a stable number.
# A gap between left and right says crashes and rallies are shaped differently.
tail_index(r).to_frame('Hill alpha')

In [ ]:
# Q: How much could a position lose over 1, 5 and 10 days?
# VaR is the loss exceeded on (1 - level) of windows; expected shortfall (ES) is the average
# loss beyond it. Historical uses the data as it is; normal assumes a bell curve; Cornish-Fisher
# corrects the normal for skew and kurtosis. Historical well above normal at 99% = fat tails.
var = value_at_risk(r)
table = var.pivot_table(index=['horizon', 'level'], columns='method', values=['var', 'es'])
table.map('{:.2%}'.format)

## 2. Dependence and volatility clustering

In [ ]:
# Q: Is there memory in direction, or in size?
# Left: returns - bars inside the band mean yesterday's direction says little about today's.
# Right: squared returns - bars well above the band, for many lags, are volatility clustering:
# big moves follow big moves, of either sign. That is what makes the rare-event
# probabilities in section 4 cluster in time, and why their intervals use a block bootstrap.
plot_autocorrelation(autocorrelation(r)).show()

In [ ]:
# Q: Is that memory statistically significant?
# Ljung-Box tests all lags up to m at once. A small p-value on `squared` confirms clustering;
# on `returns` it would mean predictable direction, which is rare for liquid markets.
lb = ljung_box(r)
lb.assign(p_value=lb['p_value'].map('{:.4f}'.format))

In [ ]:
# Q: Do moves trend or reverse over several days?
# VR(q) = 1 for a random walk. Above 1 (z_robust > 2): q-day moves are bigger than q single
# days would imply - moves persist. Below 1 (z_robust < -2): they partly reverse. This is the
# question v0.5's rolling-volatility chart raises; z_robust allows for volatility clustering.
vr = variance_ratio(r)
vr.round({'variance_ratio': 3, 'z': 2, 'z_robust': 2, 'p_value': 4})

In [ ]:
# Q: Does volatility cluster (Engle's ARCH-LM test)?
# A small p-value means today's squared return is predictable from recent ones.
arch_lm(r).to_frame('ARCH-LM (5 lags)')

## 3. Drawdowns and risk-adjusted returns

In [ ]:
# Q: How good was the return for the risk taken?
# Sharpe divides the mean return by all volatility, Sortino only by downside volatility,
# Calmar by the deepest drawdown. Annualized with trade_days.
ratios = risk_ratios(r, P.trade_days)
formats = {'annual_return': '{:.2%}', 'annual_volatility': '{:.2%}', 'max_drawdown': '{:.2%}',
           'sharpe': '{:.2f}', 'sortino': '{:.2f}', 'calmar': '{:.2f}'}
pd.Series({k: formats[k].format(v) for k, v in ratios.items()}).to_frame('value')

In [ ]:
# Q: How deep were the worst drawdowns, and how long did they take to recover?
# `days_to_recover` NaN means the drawdown is still open.
dd = drawdown_table(r, top=5)
dd.assign(depth=pct(dd['depth'], 1))

In [ ]:
# Q: When were the drawdowns, and how did risk and reward move through time?
# Underwater chart: the distance below the running peak. Rolling panels: annualized
# volatility and Sharpe over the trailing WINDOW days - where Sharpe swings from high to
# negative, the full-period ratio above averages over very different regimes.
plot_drawdown(r).show()
plot_rolling_risk(rolling_risk(r, WINDOW, P.trade_days), window=WINDOW).show()

## 4. How sure are the rare-event probabilities?

v0.5 reports the rare-event probabilities as single numbers. Here each gets a 95% interval from a
stationary block bootstrap (resampling blocks of days, so volatility clustering survives), and is
compared with what an i.i.d. normal and a fitted Student-t would predict.

In [ ]:
# Q: For the drill holding period, which rare events are well measured, and which rest on a
# handful of episodes?
# `episodes` counts each unbroken run once. A wide interval, or few episodes, means the
# probability could easily be a multiple of what is shown. Observed well above both models
# means fatter tails or more dependence than an i.i.d. model allows; between them, the
# Student-t is usually the closer.
keys = ['change_type', 'change', 'threshold', 'n_days', 'n_years']
observed = pd.concat([consecutive_analysis(r, thr, DRILL_N_DAYS, n_years, P)
                      for thr in P.return_thresholds for n_years in P.lookback_years])
events = (probability_intervals(r, DRILL_N_DAYS, P, n_boot=N_BOOT)
          .merge(model_probabilities(r, DRILL_N_DAYS, P), on=keys)
          .merge(observed[keys + ['count', 'episodes']], on=keys))

rare = events[(events['prob'] < P.prob_max) & (events['prob'] > P.prob_min)]
print(f'{len(rare)} rare events (prob between {P.prob_min:.2%} and {P.prob_max:.0%}) '
      f'for a {DRILL_N_DAYS}-day holding period')
shown = rare.sort_values(['n_years', 'change_type', 'change', 'threshold'])
shown.assign(**{c: pct(shown[c], 3) for c in ('prob', 'lower', 'upper', 'prob_normal', 'prob_t')},
             threshold=pct(shown['threshold']))[
    ['n_years', 'change_type', 'change', 'threshold', 'count', 'episodes', 'prob', 'lower',
     'upper', 'prob_normal', 'prob_t']]

In [ ]:
# Q: How do the observed probabilities compare with the models across move sizes?
# Log scale, so the rare end is visible; error bars are the 95% bootstrap intervals.
longest = max(P.lookback_years)
for change in ('above', 'below'):
    plot_event_probabilities(events, 'cumulative', change, n_years=longest).show()

## 5. Option sizing

Rebuilt from the retired `basic.py` and `test_es.ipynb`: the price band implied by recent
volatility, how often moves cleared a threshold over the last `trade_days`, and the expected P&L
per contract of the four hand-tuned grids from `test_es` (cells 33–36), unchanged.

In [ ]:
# Q: Where is the price likely to be (one standard deviation) after 1, 5 and 10 days, and by
# the close today?
# Uses the latest trailing-year daily volatility. tick=0.25 rounds outwards to the ES tick.
spot = df['price'].iloc[-1]
daily_vol = latest_snapshot(df, P)['daily_std_dev']
bands = [price_range(spot, daily_vol, days=d, tick=0.25) for d in (1, 5, 10)]
bands += [price_range(spot, daily_vol, hours=h, tick=0.25) for h in (1, 3.25)]
pd.DataFrame(bands).set_index('horizon')

In [ ]:
# Q: Over the last trade_days, how often did 1-, 5- and 10-day moves clear each threshold?
# scaled: a daily threshold x sqrt(horizon); actual: the latest rolling std of that horizon's
# return; fixed: the same threshold at every horizon. Values as used in test_es.
SCALED, FIXED = 0.01, 0.02
probs = move_probabilities(df, P, scaled=SCALED, fixed=FIXED)
probs.pivot_table(index=['horizon', 'direction'], columns='method', values='prob').map(
    '{:.1%}'.format)

In [ ]:
# Q: What did the four test_es P&L grids expect per contract?
# Each grid maps horizon -> direction -> [win if the move does not happen, loss if it does].
# Expected P&L = (1 - prob) * win + prob * loss, times the contract size.
CONTRACT_SIZE = 100
GRIDS = {
    'test_es cell 33': {1: {'above': [0.15, -0.3], 'below': [0.15, -0.3], 'exceeding': [0.2, -0.3]},
                        5: {'above': [0.2, -0.3], 'below': [0.2, -0.3], 'exceeding': [0.3, -0.3]},
                        10: {'above': [0.2, -0.3], 'below': [0.2, -0.3], 'exceeding': [0.4, -0.3]}},
    'test_es cell 34': {1: {'above': [0.15, -1], 'below': [0.15, -1], 'exceeding': [0.2, -0.9]},
                        5: {'above': [0.2, -1], 'below': [0.2, -1], 'exceeding': [0.3, -0.9]},
                        10: {'above': [0.2, -1], 'below': [0.2, -1], 'exceeding': [0.4, -0.9]}},
    'test_es cell 35': {1: {'above': [0.15, -1], 'below': [0.15, -1], 'exceeding': [0.2, -0.9]},
                        5: {'above': [0.2, -1], 'below': [0.2, -1], 'exceeding': [0.4, -0.9]},
                        10: {'above': [0.2, -1], 'below': [0.2, -1], 'exceeding': [0.4, -0.9]}},
    'test_es cell 36': {1: {'above': [0.2, -1], 'below': [0.2, -1], 'exceeding': [0.2, -0.9]},
                        5: {'above': [0.2, -1], 'below': [0.2, -1], 'exceeding': [0.45, -0.9]},
                        10: {'above': [0.2, -1], 'below': [0.2, -1], 'exceeding': [0.45, -0.9]}},
}
for name, grid in GRIDS.items():
    pnl = expected_pnl(probs, grid, CONTRACT_SIZE)
    print(name)
    display(pnl.pivot_table(index=['horizon', 'direction'], columns='method',
                            values='expected_pnl').round(2))

## Reading the results

- **Fat tails** (section 1) mean normal-based VaR and the normal model in section 4 understate
  large moves; prefer the historical or Cornish-Fisher figures, and the Student-t model.
- **Volatility clustering** (section 2) means rare events arrive in bunches: an event's
  `episodes` can be far fewer than its `count`, and its interval is wider than an i.i.d.
  calculation would suggest.
- **A rare-event probability whose interval spans a multiple of itself** (section 4) is not
  precise enough to price on alone; the models give a second opinion.
- **Expected P&L** (section 5) inherits all of the above: it uses the trailing year's move
  probabilities as if they were known exactly.